# Debugging Neural Networks: A Practical Guide

**Learning Objectives:**
- Master essential sanity checks before full training runs
- Identify and fix common failure modes (dying ReLU, NaN losses, mode collapse)
- Visualize gradient flow to diagnose training issues
- Interpret learning curves to detect overfitting, underfitting, and instability
- Build a systematic debugging workflow

**Philosophy:** Debugging neural networks is both an art and a science. This notebook teaches you to be a "neural network detective" — systematically gathering evidence, forming hypotheses, and testing solutions. You'll learn to recognize failure patterns and develop intuition for what's going wrong.

## 1. Introduction: Why Neural Networks Fail

Training neural networks is notoriously difficult. Unlike traditional software where bugs produce errors, neural networks "fail silently" — they run without crashing but produce terrible results.

**Common failure patterns:**
1. **Model doesn't learn**: Loss stays constant or decreases very slowly
2. **Training is unstable**: Loss spikes, oscillates, or diverges to NaN
3. **Model memorizes**: Perfect training accuracy but poor validation accuracy
4. **Model underfits**: Both training and validation accuracy are poor
5. **Silent bugs**: Model trains but produces nonsensical outputs

**The debugging mindset:**
- Start with the simplest possible test
- Change one thing at a time
- Visualize everything (gradients, activations, predictions)
- Trust data, not intuition
- Have realistic expectations (what's a "good" loss?)

## 2. Setup

In [ ]:
# Standard imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

Display the output.

In [ ]:
# Import shared utilities
from aiml_notebooks import get_device, set_seed

%load_ext autoreload
%autoreload 2

# Setup
set_seed(42)
device = get_device()
print(f"Using device: {device}")

## 3. Building Our Test Network

We'll create a simple multi-layer perceptron (MLP) that we can intentionally break and then debug. This will be our "patient" for diagnosis.

In [ ]:
class SimpleClassifier(nn.Module):
    """Simple MLP for binary classification.
    
    Args:
        input_dim: Input feature dimension
        hidden_dims: List of hidden layer sizes
        output_dim: Output dimension
        activation: Activation function ('relu', 'sigmoid', 'tanh')
        dropout: Dropout probability (0 = no dropout)
    """
    def __init__(self, input_dim, hidden_dims, output_dim, activation='relu', dropout=0.0):
        super().__init__()
        self.activation_name = activation
        
        # Build layers
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(self._get_activation(activation))
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(prev_dim, output_dim))
        
        self.model = nn.Sequential(*layers)
    
    def _get_activation(self, name):
        if name == 'relu':
            return nn.ReLU()
        elif name == 'sigmoid':
            return nn.Sigmoid()
        elif name == 'tanh':
            return nn.Tanh()
        else:
            raise ValueError(f"Unknown activation: {name}")
    
    def forward(self, x):
        return self.model(x)

# Test instantiation
model = SimpleClassifier(input_dim=2, hidden_dims=[64, 32], output_dim=2)
print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")
print(model)

### Create a Simple Dataset

We'll use a simple 2D classification problem — two clusters that should be easily separable. If our model can't learn this, something is seriously wrong!

In [ ]:
def generate_two_clusters(n_samples=1000, separation=3.0, noise=0.5):
    """Generate two Gaussian clusters for binary classification."""
    n = n_samples // 2
    
    # Cluster 1: centered at (-separation/2, 0)
    X1 = np.random.randn(n, 2) * noise + np.array([-separation/2, 0])
    y1 = np.zeros(n)
    
    # Cluster 2: centered at (separation/2, 0)
    X2 = np.random.randn(n, 2) * noise + np.array([separation/2, 0])
    y2 = np.ones(n)
    
    # Combine
    X = np.vstack([X1, X2])
    y = np.hstack([y1, y2])
    
    return torch.FloatTensor(X), torch.LongTensor(y)

# Generate datasets
X_train, y_train = generate_two_clusters(n_samples=1000)
X_val, y_val = generate_two_clusters(n_samples=200)

# Create dataloaders
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# Visualize
plt.figure(figsize=(8, 6))
plt.scatter(X_train[y_train==0, 0], X_train[y_train==0, 1], 
           alpha=0.6, label='Class 0', s=30)
plt.scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], 
           alpha=0.6, label='Class 1', s=30)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Training Data: Two Clusters (Should be Easy!)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Class balance: {(y_train==0).sum().item()} vs {(y_train==1).sum().item()}")

## 4. Sanity Check #1: Overfit a Small Batch

**The most important debugging test!**

Before training on the full dataset, verify your model can overfit a tiny batch (8-32 examples). This isolates the model from data issues.

**What should happen:**
- Loss should decrease to near 0
- Training accuracy should reach 100%
- This should happen within 50-200 iterations

**If it doesn't work, the problem is:**
- Model architecture (too small, wrong activation)
- Loss function (wrong choice for task)
- Optimizer (learning rate, wrong optimizer)
- Implementation bug (data/label mismatch, wrong dimensions)

In [ ]:
def sanity_check_overfit_batch(model, X, y, n_steps=200, lr=0.01):
    """Try to overfit a single small batch."""
    model = model.to(device)
    X, y = X.to(device), y.to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    losses = []
    accuracies = []
    
    for step in range(n_steps):
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        
        # Track metrics
        losses.append(loss.item())
        with torch.no_grad():
            preds = outputs.argmax(dim=1)
            acc = (preds == y).float().mean().item()
            accuracies.append(acc)
    
    return losses, accuracies

# Test on a small batch
batch_X = X_train[:16]
batch_y = y_train[:16]

print("🧪 Sanity Check: Can the model overfit 16 examples?\n")
model = SimpleClassifier(input_dim=2, hidden_dims=[64, 32], output_dim=2)
losses, accuracies = sanity_check_overfit_batch(model, batch_X, batch_y, n_steps=200)

# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(losses, linewidth=2)
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')
ax1.set_title('Loss on 16 Examples')
ax1.grid(True, alpha=0.3)

ax2.plot(accuracies, linewidth=2, color='green')
ax2.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Perfect accuracy')
ax2.set_xlabel('Step')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy on 16 Examples')
ax2.set_ylim([0, 1.05])
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Verdict
final_loss = losses[-1]
final_acc = accuracies[-1]

print(f"\n📊 Results:")
print(f"  Final loss: {final_loss:.4f}")
print(f"  Final accuracy: {final_acc:.2%}")

if final_loss < 0.1 and final_acc > 0.95:
    print("  ✅ PASS: Model can overfit! Basic architecture is working.")
elif final_loss < 0.5:
    print("  ⚠️  PARTIAL: Model is learning but slowly. Try higher learning rate.")
else:
    print("  ❌ FAIL: Model is not learning. Check architecture, loss, and learning rate.")

### Key Insight: Why This Test Matters

If your model **can't overfit 16 examples**, it will never work on the full dataset. This test eliminates:
- Data augmentation issues (we're not using any)
- Regularization issues (we're not using any)
- Dataset size issues (only 16 examples)

It isolates the core: **Can this model learn this task at all?**

## 5. Common Failure Mode #1: Dying ReLU

**Problem:** ReLU neurons that output 0 for all inputs become "dead" — their gradients are 0, so they never recover.

**Causes:**
- Learning rate too high → large weight updates → neurons move to negative region
- Bad weight initialization → many neurons start in dead state
- No batch normalization → activation distributions shift

**Symptoms:**
- Loss stops decreasing early in training
- Many activations are exactly 0
- Validation accuracy plateaus

Let's create a model with dying ReLUs!

In [ ]:
class BadInitClassifier(nn.Module):
    """Model with intentionally bad initialization to cause dying ReLU."""
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layer = nn.Linear(prev_dim, hidden_dim)
            # BAD: Initialize with large negative bias
            nn.init.constant_(layer.bias, -10.0)  # Neurons start dead!
            layers.append(layer)
            layers.append(nn.ReLU())
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, output_dim))
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x)

# Create broken model
broken_model = BadInitClassifier(input_dim=2, hidden_dims=[128, 64], output_dim=2)

# Test on small batch
print("🧪 Testing model with dying ReLU (bad initialization)\n")
losses_broken, acc_broken = sanity_check_overfit_batch(broken_model, batch_X, batch_y, n_steps=200, lr=0.01)

# Compare with healthy model
healthy_model = SimpleClassifier(input_dim=2, hidden_dims=[128, 64], output_dim=2)
losses_healthy, acc_healthy = sanity_check_overfit_batch(healthy_model, batch_X, batch_y, n_steps=200, lr=0.01)

# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(losses_broken, linewidth=2, label='Dying ReLU (broken)', color='red')
ax1.plot(losses_healthy, linewidth=2, label='Healthy model', color='green')
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')
ax1.set_title('Loss: Dying ReLU vs Healthy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(acc_broken, linewidth=2, label='Dying ReLU (broken)', color='red')
ax2.plot(acc_healthy, linewidth=2, label='Healthy model', color='green')
ax2.set_xlabel('Step')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy: Dying ReLU vs Healthy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal loss (broken): {losses_broken[-1]:.4f}")
print(f"Final loss (healthy): {losses_healthy[-1]:.4f}")

### Diagnosing Dying ReLU: Check Activations

Let's inspect the activations to see how many neurons are dead.

In [ ]:
def check_dead_neurons(model, X):
    """Count how many neurons are outputting 0 (dead)."""
    model.eval()
    X = X.to(device)
    
    activations = []
    
    def hook_fn(module, input, output):
        activations.append(output.detach().cpu())
    
    # Register hooks on ReLU layers
    hooks = []
    for module in model.modules():
        if isinstance(module, nn.ReLU):
            hooks.append(module.register_forward_hook(hook_fn))
    
    # Forward pass
    with torch.no_grad():
        _ = model(X)
    
    # Remove hooks
    for hook in hooks:
        hook.remove()
    
    # Analyze activations
    dead_percentages = []
    for i, act in enumerate(activations):
        # Neuron is dead if it outputs 0 for all samples
        dead_neurons = (act.max(dim=0).values == 0).sum().item()
        total_neurons = act.shape[1]
        dead_pct = dead_neurons / total_neurons * 100
        dead_percentages.append(dead_pct)
        print(f"Layer {i+1}: {dead_neurons}/{total_neurons} dead neurons ({dead_pct:.1f}%)")
    
    return dead_percentages

print("🔍 Checking for dead neurons:\n")
print("Broken model (bad init):")
dead_pct_broken = check_dead_neurons(broken_model, batch_X)

print("\nHealthy model:")
dead_pct_healthy = check_dead_neurons(healthy_model, batch_X)

### Solutions for Dying ReLU

1. **Use Leaky ReLU** or **PReLU**: Allows small gradient even for negative inputs
2. **Lower learning rate**: Prevents neurons from jumping into dead zone
3. **Use He initialization**: Proper weight initialization for ReLU
4. **Use Batch Normalization**: Keeps activations in healthy range
5. **Monitor dead neuron percentage**: >50% is a red flag

Let's test Leaky ReLU as a fix:

In [ ]:
class LeakyClassifier(nn.Module):
    """Model with Leaky ReLU (can't die!)."""
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layer = nn.Linear(prev_dim, hidden_dim)
            # Still using bad initialization
            nn.init.constant_(layer.bias, -10.0)
            layers.append(layer)
            layers.append(nn.LeakyReLU(0.1))  # Allows 10% gradient for negative inputs
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, output_dim))
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x)

# Test Leaky ReLU with same bad initialization
leaky_model = LeakyClassifier(input_dim=2, hidden_dims=[128, 64], output_dim=2)
losses_leaky, acc_leaky = sanity_check_overfit_batch(leaky_model, batch_X, batch_y, n_steps=200, lr=0.01)

# Compare all three
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(losses_broken, linewidth=2, label='ReLU (broken)', color='red', alpha=0.7)
ax1.plot(losses_healthy, linewidth=2, label='ReLU (healthy)', color='green', alpha=0.7)
ax1.plot(losses_leaky, linewidth=2, label='Leaky ReLU (recovers!)', color='blue', alpha=0.7)
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')
ax1.set_title('Leaky ReLU Rescues Bad Initialization')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(acc_broken, linewidth=2, label='ReLU (broken)', color='red', alpha=0.7)
ax2.plot(acc_healthy, linewidth=2, label='ReLU (healthy)', color='green', alpha=0.7)
ax2.plot(acc_leaky, linewidth=2, label='Leaky ReLU (recovers!)', color='blue', alpha=0.7)
ax2.set_xlabel('Step')
ax2.set_ylabel('Accuracy')
ax2.set_title('Leaky ReLU Enables Learning')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✅ Leaky ReLU allows neurons to recover even with bad initialization!")
print(f"   Final loss: {losses_leaky[-1]:.4f} (vs {losses_broken[-1]:.4f} for ReLU)")

## 6. Common Failure Mode #2: NaN Losses

**Problem:** Loss becomes NaN (Not a Number), training crashes.

**Common causes:**
1. **Learning rate too high** → exploding gradients → inf weights → NaN
2. **Numerical instability** in loss function (e.g., log(0))
3. **Data issues** (NaN in inputs, extreme outliers)
4. **Mixed precision training** gone wrong
5. **Incorrect loss function** for the task

Let's reproduce NaN losses and learn how to diagnose them.

In [ ]:
def train_until_nan(model, X, y, lr=10.0, max_steps=100):
    """Train with very high LR until NaN appears."""
    model = model.to(device)
    X, y = X.to(device), y.to(device)
    
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    losses = []
    gradient_norms = []
    
    for step in range(max_steps):
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        
        # Check for NaN
        if torch.isnan(loss):
            print(f"❌ NaN loss detected at step {step}!")
            break
        
        losses.append(loss.item())
        
        loss.backward()
        
        # Track gradient norm
        total_norm = 0
        for p in model.parameters():
            if p.grad is not None:
                total_norm += p.grad.data.norm(2).item() ** 2
        total_norm = total_norm ** 0.5
        gradient_norms.append(total_norm)
        
        optimizer.step()
        
        # Early warning
        if loss.item() > 1e6:
            print(f"⚠️  Warning: Loss exploding at step {step} (loss={loss.item():.2e})")
    
    return losses, gradient_norms

# Create model and intentionally break it
print("🧪 Reproducing NaN loss (learning rate = 10.0)\n")
model_nan = SimpleClassifier(input_dim=2, hidden_dims=[64, 32], output_dim=2)
losses_nan, grads_nan = train_until_nan(model_nan, batch_X, batch_y, lr=10.0)

# Visualize the explosion
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(losses_nan, linewidth=2, color='red', marker='o', markersize=4)
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Explosion → NaN')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

ax2.plot(grads_nan, linewidth=2, color='orange', marker='s', markersize=4)
ax2.set_xlabel('Step')
ax2.set_ylabel('Gradient Norm')
ax2.set_title('Gradient Explosion (Precedes NaN)')
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Loss progression:")
for i, loss in enumerate(losses_nan[:10]):
    print(f"  Step {i}: {loss:.4f}")
if len(losses_nan) > 10:
    print(f"  ...")
    print(f"  Step {len(losses_nan)-1}: {losses_nan[-1]:.2e}")

### Debugging NaN Losses: Step-by-Step

When you encounter NaN losses, follow this checklist:

In [ ]:
def diagnose_nan_losses(model, X, y, lr=0.01):
    """Systematic NaN debugging checklist."""
    print("🔍 NaN Loss Diagnostic Checklist\n")
    print("="*60)
    
    # 1. Check input data
    print("\n1. Checking input data:")
    has_nan_input = torch.isnan(X).any().item()
    has_inf_input = torch.isinf(X).any().item()
    print(f"   NaN in inputs: {'❌ YES' if has_nan_input else '✅ NO'}")
    print(f"   Inf in inputs: {'❌ YES' if has_inf_input else '✅ NO'}")
    print(f"   Input range: [{X.min().item():.2f}, {X.max().item():.2f}]")
    
    # 2. Check labels
    print("\n2. Checking labels:")
    print(f"   Label range: [{y.min().item()}, {y.max().item()}]")
    print(f"   Expected range: [0, num_classes-1]")
    
    # 3. Check model weights
    print("\n3. Checking model initialization:")
    for name, param in model.named_parameters():
        has_nan = torch.isnan(param).any().item()
        has_inf = torch.isinf(param).any().item()
        if has_nan or has_inf:
            print(f"   ❌ {name}: NaN={has_nan}, Inf={has_inf}")
    print("   ✅ All weights initialized properly")
    
    # 4. Check forward pass
    print("\n4. Checking forward pass:")
    model.eval()
    with torch.no_grad():
        outputs = model(X.to(device))
        has_nan_output = torch.isnan(outputs).any().item()
        has_inf_output = torch.isinf(outputs).any().item()
        print(f"   NaN in outputs: {'❌ YES' if has_nan_output else '✅ NO'}")
        print(f"   Inf in outputs: {'❌ YES' if has_inf_output else '✅ NO'}")
        print(f"   Output range: [{outputs.min().item():.2f}, {outputs.max().item():.2f}]")
    
    # 5. Check loss computation
    print("\n5. Checking loss computation:")
    criterion = nn.CrossEntropyLoss()
    loss = criterion(outputs.cpu(), y)
    print(f"   Initial loss: {loss.item():.4f}")
    print(f"   Loss is NaN: {'❌ YES' if torch.isnan(loss) else '✅ NO'}")
    
    # 6. Check gradient computation
    print("\n6. Checking gradient computation:")
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    optimizer.zero_grad()
    outputs = model(X.to(device))
    loss = criterion(outputs, y.to(device))
    loss.backward()
    
    max_grad = 0
    has_nan_grad = False
    for param in model.parameters():
        if param.grad is not None:
            if torch.isnan(param.grad).any():
                has_nan_grad = True
            max_grad = max(max_grad, param.grad.abs().max().item())
    
    print(f"   NaN in gradients: {'❌ YES' if has_nan_grad else '✅ NO'}")
    print(f"   Max gradient magnitude: {max_grad:.2e}")
    
    if max_grad > 100:
        print(f"   ⚠️  Large gradients detected! Consider:")
        print(f"      - Lower learning rate (current: {lr})")
        print(f"      - Gradient clipping")
        print(f"      - Better weight initialization")
    
    print("\n" + "="*60)

# Run diagnostics
model_test = SimpleClassifier(input_dim=2, hidden_dims=[64, 32], output_dim=2)
diagnose_nan_losses(model_test, batch_X, batch_y, lr=10.0)

### Solutions for NaN Losses

1. **Reduce learning rate** (most common fix)
2. **Use gradient clipping** (e.g., `torch.nn.utils.clip_grad_norm_()`)
3. **Check data preprocessing** (normalize inputs, check for NaNs)
4. **Use numerically stable loss functions** (PyTorch's built-ins are usually safe)
5. **Enable anomaly detection** for debugging:
   ```python
   torch.autograd.set_detect_anomaly(True)
   ```

## 7. Gradient Flow Visualization

Visualizing gradients helps identify:
- **Vanishing gradients**: Early layers have tiny gradients
- **Exploding gradients**: Some layers have huge gradients
- **Dead layers**: Layers with zero gradients
- **Uneven flow**: Some layers learn much faster than others

Let's build a gradient monitoring system.

In [ ]:
class GradientMonitor:
    """Monitor gradient flow during training."""
    def __init__(self, model):
        self.model = model
        self.gradient_history = defaultdict(list)
    
    def record_gradients(self):
        """Record gradient norms for each layer."""
        for name, param in self.model.named_parameters():
            if param.grad is not None:
                grad_norm = param.grad.norm().item()
                self.gradient_history[name].append(grad_norm)
    
    def plot_gradient_flow(self, title="Gradient Flow"):
        """Plot gradient evolution over training."""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
        
        # Line plot: gradient evolution
        for name, grads in self.gradient_history.items():
            if 'weight' in name:  # Only plot weights, not biases
                ax1.plot(grads, label=name, alpha=0.7, linewidth=2)
        
        ax1.set_xlabel('Training Step', fontsize=11)
        ax1.set_ylabel('Gradient Norm', fontsize=11)
        ax1.set_title(f'{title} - Evolution', fontsize=12, fontweight='bold')
        ax1.set_yscale('log')
        ax1.legend(fontsize=8, loc='best')
        ax1.grid(True, alpha=0.3)
        
        # Box plot: final gradient distribution
        weight_names = [name for name in self.gradient_history.keys() if 'weight' in name]
        final_grads = [self.gradient_history[name][-50:] for name in weight_names]
        
        ax2.boxplot(final_grads, labels=[name.split('.')[1] for name in weight_names])
        ax2.set_xlabel('Layer', fontsize=11)
        ax2.set_ylabel('Gradient Norm (last 50 steps)', fontsize=11)
        ax2.set_title(f'{title} - Distribution', fontsize=12, fontweight='bold')
        ax2.set_yscale('log')
        ax2.grid(True, alpha=0.3, axis='y')
        plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')
        
        plt.tight_layout()
        plt.show()

# Train a model while monitoring gradients
def train_with_gradient_monitoring(model, train_loader, epochs=5, lr=0.01):
    """Train and monitor gradient flow."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    monitor = GradientMonitor(model)
    
    losses = []
    
    for epoch in range(epochs):
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            
            # Record gradients before optimizer step
            monitor.record_gradients()
            
            optimizer.step()
            losses.append(loss.item())
    
    return losses, monitor

print("🔍 Training with gradient monitoring...\n")
model = SimpleClassifier(input_dim=2, hidden_dims=[128, 64, 32], output_dim=2)
losses, monitor = train_with_gradient_monitoring(model, train_loader, epochs=3, lr=0.01)

monitor.plot_gradient_flow(title="Healthy Model")

### Interpreting Gradient Flow Plots

**Healthy gradient flow:**
- All layers have similar magnitude gradients (within 1-2 orders of magnitude)
- Gradients remain stable over training
- No layers have consistently zero gradients

**Problem patterns:**
- **Vanishing**: Early layers have gradients 100-1000x smaller than late layers
- **Exploding**: Gradients grow exponentially during training
- **Dead layers**: Flat lines at or near zero
- **Unstable**: Wild oscillations in gradient magnitude

## 8. Learning Curve Interpretation

Learning curves (plots of loss vs epoch) tell a story about your model's training. Let's learn to read them!

In [ ]:
def train_and_evaluate(model, train_loader, val_loader, epochs=20, lr=0.01):
    """Train model and track both train and validation metrics."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': []
    }
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * X_batch.size(0)
            preds = outputs.argmax(dim=1)
            train_correct += (preds == y_batch).sum().item()
            train_total += X_batch.size(0)
        
        # Validation
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                
                val_loss += loss.item() * X_batch.size(0)
                preds = outputs.argmax(dim=1)
                val_correct += (preds == y_batch).sum().item()
                val_total += X_batch.size(0)
        
        # Record metrics
        history['train_loss'].append(train_loss / train_total)
        history['val_loss'].append(val_loss / val_total)
        history['train_acc'].append(train_correct / train_total)
        history['val_acc'].append(val_correct / val_total)
    
    return history

# Helper function to plot learning curves
def plot_learning_curves(history, title="Learning Curves"):
    """Plot training and validation metrics."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Loss
    ax1.plot(epochs, history['train_loss'], 'o-', label='Training', linewidth=2, markersize=4)
    ax1.plot(epochs, history['val_loss'], 's-', label='Validation', linewidth=2, markersize=4)
    ax1.set_xlabel('Epoch', fontsize=11)
    ax1.set_ylabel('Loss', fontsize=11)
    ax1.set_title(f'{title} - Loss', fontsize=12, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    # Accuracy
    ax2.plot(epochs, history['train_acc'], 'o-', label='Training', linewidth=2, markersize=4)
    ax2.plot(epochs, history['val_acc'], 's-', label='Validation', linewidth=2, markersize=4)
    ax2.set_xlabel('Epoch', fontsize=11)
    ax2.set_ylabel('Accuracy', fontsize=11)
    ax2.set_title(f'{title} - Accuracy', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Train a healthy model
print("📈 Scenario 1: Healthy Training\n")
model_healthy = SimpleClassifier(input_dim=2, hidden_dims=[64, 32], output_dim=2)
history_healthy = train_and_evaluate(model_healthy, train_loader, val_loader, epochs=20, lr=0.01)
plot_learning_curves(history_healthy, title="Healthy Model")

print(f"Final train accuracy: {history_healthy['train_acc'][-1]:.2%}")
print(f"Final val accuracy: {history_healthy['val_acc'][-1]:.2%}")
print(f"Train-val gap: {(history_healthy['train_acc'][-1] - history_healthy['val_acc'][-1]):.2%}")

### Learning Curve Pattern #1: Overfitting

**Symptoms:**
- Training loss keeps decreasing
- Validation loss starts increasing after some point
- Large gap between train and validation accuracy

Let's create an overfitting scenario:

In [ ]:
print("📈 Scenario 2: Overfitting (Large Model, No Regularization)\n")

# Very large model for simple task
model_overfit = SimpleClassifier(input_dim=2, hidden_dims=[256, 256, 128, 64], output_dim=2)
history_overfit = train_and_evaluate(model_overfit, train_loader, val_loader, epochs=40, lr=0.01)
plot_learning_curves(history_overfit, title="Overfitting Model")

print(f"Final train accuracy: {history_overfit['train_acc'][-1]:.2%}")
print(f"Final val accuracy: {history_overfit['val_acc'][-1]:.2%}")
print(f"Train-val gap: {(history_overfit['train_acc'][-1] - history_overfit['val_acc'][-1]):.2%}")
print("\n⚠️  Large train-val gap indicates overfitting!")

### Learning Curve Pattern #2: Underfitting

**Symptoms:**
- Both training and validation loss are high
- Loss plateaus quickly
- Small gap between train and validation
- Poor performance on both sets

In [ ]:
print("📈 Scenario 3: Underfitting (Model Too Simple)\n")

# Very small model
model_underfit = SimpleClassifier(input_dim=2, hidden_dims=[4], output_dim=2)
history_underfit = train_and_evaluate(model_underfit, train_loader, val_loader, epochs=20, lr=0.01)
plot_learning_curves(history_underfit, title="Underfitting Model")

print(f"Final train accuracy: {history_underfit['train_acc'][-1]:.2%}")
print(f"Final val accuracy: {history_underfit['val_acc'][-1]:.2%}")
print(f"Train-val gap: {(history_underfit['train_acc'][-1] - history_underfit['val_acc'][-1]):.2%}")
print("\n⚠️  Low training accuracy indicates underfitting!")

### Learning Curve Pattern #3: Learning Rate Too High

**Symptoms:**
- Loss oscillates wildly
- No smooth decrease in loss
- May diverge or plateau at suboptimal value

In [ ]:
print("📈 Scenario 4: Learning Rate Too High\n")

model_high_lr = SimpleClassifier(input_dim=2, hidden_dims=[64, 32], output_dim=2)
history_high_lr = train_and_evaluate(model_high_lr, train_loader, val_loader, epochs=20, lr=0.5)  # Very high LR
plot_learning_curves(history_high_lr, title="Learning Rate Too High")

print("\n⚠️  Oscillating loss indicates learning rate is too high!")

### Comparative Visualization: All Scenarios

Let's compare all failure modes side-by-side:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

scenarios = [
    (history_healthy, "Healthy Training", "green"),
    (history_overfit, "Overfitting", "red"),
    (history_underfit, "Underfitting", "orange"),
    (history_high_lr, "LR Too High", "purple")
]

for idx, (history, title, color) in enumerate(scenarios):
    row = idx // 2
    col = idx % 2
    ax = axes[row, col]
    
    epochs = range(1, len(history['train_loss']) + 1)
    ax.plot(epochs, history['train_loss'], 'o-', label='Train Loss', 
           linewidth=2, markersize=3, alpha=0.7, color=color)
    ax.plot(epochs, history['val_loss'], 's--', label='Val Loss', 
           linewidth=2, markersize=3, alpha=0.7, color=color)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(title, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🔍 How to identify each pattern:")
print("\n1. Healthy: Both curves decrease smoothly, small final gap")
print("2. Overfitting: Train keeps improving, val starts increasing")
print("3. Underfitting: Both curves plateau early at high loss")
print("4. LR too high: Erratic oscillations, no smooth descent")

## 9. Debugging Checklist: Systematic Workflow

When your model isn't working, follow this step-by-step debugging workflow.

### The Complete Debugging Checklist

**Phase 1: Data Sanity Checks**
- [ ] Print shapes of inputs and labels
- [ ] Visualize a few samples (do they look correct?)
- [ ] Check for NaN/Inf in inputs
- [ ] Verify label range (0 to num_classes-1 for classification)
- [ ] Check class balance (severe imbalance needs handling)
- [ ] Verify data normalization (mean ≈ 0, std ≈ 1)

**Phase 2: Model Sanity Checks**
- [ ] Overfit a small batch (8-32 examples) → should reach ~100% accuracy
- [ ] Check model capacity (too small = underfit, too large = overfit)
- [ ] Verify forward pass output shape matches expected
- [ ] Check for dead neurons (>50% = problem)
- [ ] Visualize gradient flow (all layers should have similar magnitude)

**Phase 3: Training Sanity Checks**
- [ ] Loss decreases on first batch? (If not, LR too low or model broken)
- [ ] Try random predictions baseline (your model should beat this)
- [ ] Check learning rate (start with 1e-3 or 1e-4)
- [ ] Monitor gradient norms (1e-5 to 1e1 is healthy range)
- [ ] Watch for NaN losses (exploding gradients)

**Phase 4: Interpreting Learning Curves**
- [ ] Train loss decreasing? (No = model can't learn)
- [ ] Val loss decreasing? (No = overfitting or data leak)
- [ ] Gap between train and val? (Large = overfitting)
- [ ] Both losses high? (Model underfitting)
- [ ] Loss oscillating? (LR too high)

**Phase 5: Common Fixes**
- [ ] Underfitting → Increase model size, train longer
- [ ] Overfitting → Add regularization (dropout, weight decay), get more data
- [ ] Dying ReLU → Use Leaky ReLU, lower LR, check initialization
- [ ] Exploding gradients → Lower LR, gradient clipping, better init
- [ ] Vanishing gradients → Use ReLU, add batch norm, residual connections

## 10. Practical Debugging Session

Let's debug a broken model together, step by step.

In [ ]:
print("🐛 Debugging Session: Mystery Model\n")
print("Your colleague gives you this model and says 'it doesn't work'.")
print("Let's debug it systematically!\n")

# Broken model from colleague
mystery_model = SimpleClassifier(
    input_dim=2, 
    hidden_dims=[8],  # Too small?
    output_dim=2, 
    activation='sigmoid'  # Problematic activation?
)

print("="*60)
print("STEP 1: Check model on small batch")
print("="*60)
losses_mystery, acc_mystery = sanity_check_overfit_batch(
    mystery_model, batch_X, batch_y, n_steps=500, lr=0.1
)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(losses_mystery, linewidth=2)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Mystery Model - Loss')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(acc_mystery, linewidth=2, color='green')
plt.xlabel('Step')
plt.ylabel('Accuracy')
plt.title('Mystery Model - Accuracy')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n🔍 Diagnosis:")
print(f"   Final loss: {losses_mystery[-1]:.4f}")
print(f"   Final accuracy: {acc_mystery[-1]:.2%}")

if acc_mystery[-1] < 0.9:
    print(f"\n❌ Model can't even overfit 16 examples!")
    print(f"\n💡 Hypotheses:")
    print(f"   1. Model too small (only {sum(p.numel() for p in mystery_model.parameters())} params)")
    print(f"   2. Sigmoid activation (vanishing gradients)")
    print(f"   3. Learning rate issues")

Visualize the results.

In [ ]:
print("\n" + "="*60)
print("STEP 2: Test fixes one by one")
print("="*60)

# Fix 1: Larger model
print("\n🔧 Fix 1: Increase model size")
fixed_model_1 = SimpleClassifier(input_dim=2, hidden_dims=[64, 32], output_dim=2, activation='sigmoid')
losses_fix1, acc_fix1 = sanity_check_overfit_batch(fixed_model_1, batch_X, batch_y, n_steps=200, lr=0.1)
print(f"   Result: Loss={losses_fix1[-1]:.4f}, Acc={acc_fix1[-1]:.2%}")

# Fix 2: Better activation
print("\n🔧 Fix 2: Use ReLU instead of sigmoid")
fixed_model_2 = SimpleClassifier(input_dim=2, hidden_dims=[8], output_dim=2, activation='relu')
losses_fix2, acc_fix2 = sanity_check_overfit_batch(fixed_model_2, batch_X, batch_y, n_steps=200, lr=0.1)
print(f"   Result: Loss={losses_fix2[-1]:.4f}, Acc={acc_fix2[-1]:.2%}")

# Fix 3: Both!
print("\n🔧 Fix 3: Larger model + ReLU")
fixed_model_3 = SimpleClassifier(input_dim=2, hidden_dims=[64, 32], output_dim=2, activation='relu')
losses_fix3, acc_fix3 = sanity_check_overfit_batch(fixed_model_3, batch_X, batch_y, n_steps=200, lr=0.1)
print(f"   Result: Loss={losses_fix3[-1]:.4f}, Acc={acc_fix3[-1]:.2%}")

# Compare all fixes
plt.figure(figsize=(12, 5))
plt.plot(losses_mystery[:200], label='Original (broken)', linewidth=2, alpha=0.7)
plt.plot(losses_fix1, label='Fix 1: Larger model', linewidth=2, alpha=0.7)
plt.plot(losses_fix2, label='Fix 2: ReLU', linewidth=2, alpha=0.7)
plt.plot(losses_fix3, label='Fix 3: Both', linewidth=2, alpha=0.7, color='green')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Comparing Fixes', fontweight='bold', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\n✅ Conclusion: The model was both too small AND using wrong activation!")
print("   Lesson: Test hypotheses one at a time to isolate the problem.")

## 11. Key Takeaways

### 🎓 What You've Learned

**1. Essential Debugging Philosophy**
- Start simple: overfit small batch first
- Change one thing at a time
- Visualize everything (data, predictions, gradients, curves)
- Have realistic expectations

**2. Common Failure Modes**
- **Dying ReLU**: Neurons output 0 → Fix with Leaky ReLU, better init, lower LR
- **NaN losses**: Exploding gradients → Fix with lower LR, gradient clipping
- **Overfitting**: Train ≫ val accuracy → Fix with regularization, more data
- **Underfitting**: Both train and val poor → Fix with larger model, train longer

**3. Critical Sanity Checks**
- ✅ Can model overfit 16 examples? (If no, architecture is broken)
- ✅ Are gradients flowing? (Check gradient norms)
- ✅ Is loss decreasing on first batch? (If no, LR or loss function issue)
- ✅ Do learning curves make sense? (Compare train vs val)

**4. Interpreting Learning Curves**

| Pattern | Train Loss | Val Loss | Diagnosis | Fix |
|---------|-----------|----------|-----------|-----|
| Healthy | ↓ steady | ↓ steady | Working! | Keep training |
| Overfit | ↓ steady | ↑ or plateau | Memorizing | Regularization |
| Underfit | ↓ slow, plateau | ↓ slow, plateau | Too simple | Bigger model |
| LR too high | Oscillates | Oscillates | Unstable | Lower LR |
| LR too low | ↓ very slow | ↓ very slow | Slow learning | Higher LR |

**5. Debugging Workflow**
1. Check data (shapes, visualization, no NaN/Inf)
2. Overfit small batch (must work!)
3. Monitor gradients (healthy range: 1e-5 to 1e1)
4. Inspect learning curves (compare train vs val)
5. Test fixes one at a time

### 🚀 Next Steps

- Apply these techniques to your own projects
- Build debugging into your training pipelines from day 1
- Create reusable debugging utilities (gradient monitors, sanity checks)
- Learn to use advanced tools: TensorBoard, Weights & Biases

### 📚 Further Reading

- [A Recipe for Training Neural Networks (Andrej Karpathy)](http://karpathy.github.io/2019/04/25/recipe/)
- [Troubleshooting Deep Neural Networks (Josh Tobin)](http://josh-tobin.com/troubleshooting-deep-neural-networks)
- [Deep Learning Tuning Playbook (Google Research)](https://github.com/google-research/tuning_playbook)

## 12. Practice Exercise: Debug This Model!

Apply what you've learned. Here's a broken model — debug it!

In [ ]:
# Challenge: This model has multiple problems. Find and fix them!

class BrokenModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 500)
        self.fc2 = nn.Linear(500, 500)
        self.fc3 = nn.Linear(500, 500)
        self.fc4 = nn.Linear(500, 2)
        
        # Intentional bad initialization
        for layer in [self.fc1, self.fc2, self.fc3, self.fc4]:
            nn.init.constant_(layer.weight, 0.0)
            nn.init.constant_(layer.bias, 0.0)
    
    def forward(self, x):
        x = torch.sigmoid(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        x = self.fc4(x)
        return x

print("🎯 Your challenge: Debug this model!\n")
print("Problems to find:")
print("1. What's wrong with the initialization?")
print("2. What's wrong with the activation functions?")
print("3. What's wrong with the architecture?")
print("\nHint: Try the small batch overfit test first!")

# Uncomment to test:
# broken_challenge = BrokenModel()
# losses_challenge, acc_challenge = sanity_check_overfit_batch(
#     broken_challenge, batch_X, batch_y, n_steps=300, lr=0.01
# )
# print(f"\nResult: Loss={losses_challenge[-1]:.4f}, Acc={acc_challenge[-1]:.2%}")

## Congratulations! 🎉

You've completed the neural network debugging guide. You now know how to:
- ✅ Systematically debug failing neural networks
- ✅ Identify and fix common failure modes (dying ReLU, NaN losses)
- ✅ Use sanity checks to isolate problems
- ✅ Visualize gradient flow to diagnose training issues
- ✅ Interpret learning curves to understand model behavior

**Remember:** Debugging is a skill that improves with practice. Every broken model teaches you something new. Stay curious, stay systematic, and trust the data!

Happy debugging! 🐛🔧